# 8. Option Pricing: Monte Carlo vs Black-Scholes - Interactive

**Objective**: Understand how option prices change with different market parameters using the Black-Scholes formula and Monte Carlo simulation.

## What You'll Learn
- **Black-Scholes Formula**: Exact pricing for European options
- **Monte Carlo Method**: Numerical simulation approach
- **Greeks**: Sensitivity to different parameters (Vega, Theta, Delta)
- **Put-Call Parity**: Arbitrage relationship between calls and puts
- **How Parameters Affect Prices**: Interactive exploration

## The Black-Scholes Formula
$$C = S_0 N(d_1) - K e^{-rT} N(d_2)$$

where:
- $d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}$
- $d_2 = d_1 - \sigma\sqrt{T}$
- $N(x)$ = cumulative normal distribution

In [2]:
# Import Required Libraries
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Output
    widgets_available = True
except ImportError:
    widgets_available = False
    print("⚠ ipywidgets not available. Please install: pip install ipywidgets")

# Set matplotlib to use inline display
%matplotlib inline

print("✓ Libraries loaded successfully")

✓ Libraries loaded successfully


In [3]:
# Define Pricing Functions

def black_scholes_pricing(S0, K, r, sigma, T):
    """
    Black-Scholes formula for European option pricing.
    
    Parameters:
    - S0: Current stock price
    - K: Strike price
    - r: Risk-free rate
    - sigma: Volatility (std dev of returns)
    - T: Time to maturity (years)
    
    Returns:
    - call_price: European call option price
    - put_price: European put option price
    """
    # Avoid division by zero
    if T <= 0:
        return max(S0 - K, 0), max(K - S0, 0)
    
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    call_price = S0 * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)
    put_price = K * np.exp(-r * T) * stats.norm.cdf(-d2) - S0 * stats.norm.cdf(-d1)
    
    return call_price, put_price


def monte_carlo_pricing(S0, K, r, sigma, T, N=10000):
    """
    Monte Carlo simulation for option pricing.
    
    Parameters:
    - S0: Current stock price
    - K: Strike price
    - r: Risk-free rate
    - sigma: Volatility
    - T: Time to maturity (years)
    - N: Number of simulations
    
    Returns:
    - call_price: Estimated call price
    - put_price: Estimated put price
    """
    np.random.seed(42)
    dt = T / 252  # Daily steps
    steps = int(T * 252)
    
    # Generate geometric Brownian motion paths
    Z = np.random.standard_normal((steps, N))
    paths = np.zeros((steps + 1, N))
    paths[0, :] = S0
    
    for t in range(1, steps + 1):
        paths[t, :] = paths[t-1, :] * np.exp((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z[t-1, :])
    
    # Calculate payoffs at maturity
    final_prices = paths[-1, :]
    call_payoff = np.maximum(final_prices - K, 0)
    put_payoff = np.maximum(K - final_prices, 0)
    
    # Discount back to present
    call_price = np.exp(-r * T) * np.mean(call_payoff)
    put_price = np.exp(-r * T) * np.mean(put_payoff)
    
    return call_price, put_price


print("✓ Pricing functions defined")

✓ Pricing functions defined


In [4]:
# Define Visualization Function

def display_pricing_comparison(S0, K, r, sigma, T, show_mc=False):
    """
    Visualize Black-Scholes option prices and sensitivities.
    
    Parameters:
    - S0: Current stock price
    - K: Strike price
    - r: Risk-free rate
    - sigma: Volatility
    - T: Time to maturity
    - show_mc: Whether to show Monte Carlo estimates
    """
    
    # Generate range of stock prices for plotting
    stock_prices = np.linspace(K * 0.5, K * 1.5, 100)
    
    # Calculate option prices for different stock prices
    call_prices_bs = []
    put_prices_bs = []
    
    for S in stock_prices:
        c, p = black_scholes_pricing(S, K, r, sigma, T)
        call_prices_bs.append(c)
        put_prices_bs.append(p)
    
    # Get current prices
    current_call_bs, current_put_bs = black_scholes_pricing(S0, K, r, sigma, T)
    
    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Call Option Plot
    ax1.plot(stock_prices, call_prices_bs, color='#1f77b4', linewidth=2.5, label='Call Price (BS)')
    ax1.axvline(K, color='red', linewidth=2, linestyle='--', alpha=0.7, label=f'Strike = ${K}')
    ax1.plot(S0, current_call_bs, 'go', markersize=12, label=f'Current: ${current_call_bs:.4f}', zorder=5)
    ax1.axhline(0, color='black', linewidth=0.8, alpha=0.3)
    ax1.set_xlabel('Stock Price at Maturity ($)', fontsize=11)
    ax1.set_ylabel('Call Option Price ($)', fontsize=11)
    ax1.set_title(f'Call Option Price (K=${K}, σ={sigma:.2f}, T={T:.2f}y, r={r:.2%})', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(alpha=0.3)
    
    # Put Option Plot
    ax2.plot(stock_prices, put_prices_bs, color='#ff7f0e', linewidth=2.5, label='Put Price (BS)')
    ax2.axvline(K, color='red', linewidth=2, linestyle='--', alpha=0.7, label=f'Strike = ${K}')
    ax2.plot(S0, current_put_bs, 'go', markersize=12, label=f'Current: ${current_put_bs:.4f}', zorder=5)
    ax2.axhline(0, color='black', linewidth=0.8, alpha=0.3)
    ax2.set_xlabel('Stock Price at Maturity ($)', fontsize=11)
    ax2.set_ylabel('Put Option Price ($)', fontsize=11)
    ax2.set_title(f'Put Option Price (K=${K}, σ={sigma:.2f}, T={T:.2f}y, r={r:.2%})', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate and display analysis
    print("\n" + "="*70)
    print(f"BLACK-SCHOLES PRICING ANALYSIS")
    print("="*70)
    print(f"Current Stock Price (S₀):     ${S0:.2f}")
    print(f"Strike Price (K):             ${K:.2f}")
    print(f"Time to Maturity (T):         {T:.3f} years ({T*365:.0f} days)")
    print(f"Risk-free Rate (r):           {r:.2%}")
    print(f"Volatility (σ):               {sigma:.2%}")
    print("-"*70)
    print(f"Call Option Price:            ${current_call_bs:.4f}")
    print(f"Put Option Price:             ${current_put_bs:.4f}")
    print(f"Put-Call Parity (C - P):      ${current_call_bs - current_put_bs:.4f}")
    print(f"Intrinsic Value (S - K):      ${S0 - K:.4f}")
    print(f"Verification (C-P vs S-K*e):  ${(current_call_bs - current_put_bs) - (S0 - K*np.exp(-r*T)):.6f}")
    
    if show_mc and T > 0:
        mc_call, mc_put = monte_carlo_pricing(S0, K, r, sigma, T, N=10000)
        print("-"*70)
        print(f"Monte Carlo Verification (10K paths):")
        print(f"  MC Call Price:  ${mc_call:.4f} (Error: ${abs(mc_call - current_call_bs):.6f})")
        print(f"  MC Put Price:   ${mc_put:.4f} (Error: ${abs(mc_put - current_put_bs):.6f})")
    
    print("="*70 + "\n")

print("✓ Visualization function defined")

✓ Visualization function defined


## Interactive Option Pricing Explorer

**Use the sliders below to explore how option prices change with different market parameters:**

- **Stock Price (S₀)**: Current price of the underlying stock [50-200]
- **Strike Price (K)**: Price at which you can buy (call) or sell (put) [50-200]
- **Volatility (σ)**: How much the stock price fluctuates (annual standard deviation) [5%-50%]
- **Time to Maturity (T)**: How long until the option expires [0.1-5 years]
- **Risk-free Rate (r)**: Return from a safe investment like Treasury bonds [0%-10%]
- **Show Monte Carlo**: Check to see simulation-based pricing alongside Black-Scholes

**Watch how:**
- ↑ Volatility → ↑ Both call and put prices (more uncertainty = more value)
- ↓ Time to maturity → ↓ Option prices (time value decay)
- Stock price changes shift the curves and current price point
- Different strike prices create different moneyness scenarios

In [ ]:
# Interactive Parameter Exploration
if widgets_available:
    interact(
        display_pricing_comparison,
        S0=FloatSlider(value=100, min=50, max=200, step=5, description='Stock Price (S₀)', style={'description_width': '140px'}),
        K=FloatSlider(value=100, min=50, max=200, step=5, description='Strike Price (K)', style={'description_width': '140px'}),
        r=FloatSlider(value=0.05, min=0.0, max=0.1, step=0.01, description='Risk-free Rate (r)', style={'description_width': '140px'}),
        sigma=FloatSlider(value=0.2, min=0.05, max=0.5, step=0.05, description='Volatility (σ)', style={'description_width': '140px'}),
        T=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Time to Maturity (T)', style={'description_width': '140px'}),
        show_mc=False
    )
else:
    print("⚠ Interactive sliders not available. Using static visualization instead.")
    display_pricing_comparison(100, 100, 0.05, 0.2, 1.0, show_mc=False)

interactive(children=(FloatSlider(value=100.0, description='Stock Price (S₀)', max=200.0, min=50.0, step=5.0, …

# 8. Option Pricing: Monte Carlo vs Black-Scholes

- **Objective**: Price both Calls and Puts using Monte Carlo, compare to exact formulas, and verify Put-Call Parity.

## Guided Learning Experiments

**Experiment 1: Volatility Effect (Vega)**
- Keep everything at default (S₀=100, K=100, T=1, r=5%)
- Slowly increase σ from 5% to 50%
- **Observation**: Both call and put prices increase as volatility rises
- **Why?**: Higher uncertainty increases the value of both rights to buy and sell

**Experiment 2: Time Decay (Theta)**
- Keep S₀=100, K=100, σ=20%, r=5%
- Decrease T from 5 years to 0.1 years
- **Observation**: Option prices decrease as maturity approaches, especially out-of-the-money
- **Why?**: Less time for favorable price moves = less value (time value decay)

**Experiment 3: Moneyness Scenarios**
- Keep T=1, σ=20%, r=5%
- Try three strikes: K=80 (ITM call), K=100 (ATM), K=120 (OTM call)
- **Observation**: Call prices vary dramatically; put prices are highest for low strikes
- **Why?**: Moneyness (relationship between S and K) is the key price driver

**Experiment 4: Interest Rate Impact (Rho)**
- Keep S₀=100, K=100, T=1, σ=20%
- Change r from 0% to 10%
- **Observation**: Call prices increase slightly, put prices decrease slightly
- **Why?**: Higher rates make future strike payments less expensive (lower discount)

**Experiment 5: Put-Call Parity Verification**
- Try any combination of parameters
- Check the "Verification" line in the output
- **Expected**: The value should be very close to 0 (within $0.001)
- **Formula**: C - P should equal S₀ - K × e^(-rT)
- **Why?**: This is an arbitrage relationship—a market principle, not a formula

## Key Insights & The Greeks

### The Four Greeks (Sensitivities)

| Greek | What It Measures | Effect | Intuition |
|-------|------------------|--------|-----------|
| **Delta (Δ)** | Price change per $1 stock move | Call: 0 to 1, Put: -1 to 0 | How fast option value changes |
| **Vega (ν)** | Price change per 1% volatility change | Always positive for both | More uncertainty = higher prices |
| **Theta (Θ)** | Price change per day of time passing | Negative for both | Time decay works against holder |
| **Rho (ρ)** | Price change per 1% interest rate change | Positive for calls, negative for puts | Rate affects discount factor |

### Key Insights from Black-Scholes

1. **Volatility is Everything**: Option prices rise with volatility more than any other factor
2. **Time Works Against Buyers**: As expiration approaches, time value decays to zero
3. **Put-Call Parity**: C - P = S₀ - K·e^(-rT) always holds (no arbitrage)
4. **European vs American**: Black-Scholes is for European options (only exercise at maturity)
5. **Lognormal Distribution**: Assumes stock prices follow a geometric Brownian motion
6. **Perfect Markets**: Assumes no transaction costs, no taxes, constant volatility, no dividends

### Quick Reference Table

| Condition | Call Price | Put Price |
|-----------|-----------|-----------|
| ↑ Stock Price | ↑ | ↓ |
| ↑ Strike Price | ↓ | ↑ |
| ↑ Volatility | ↑ | ↑ |
| ↑ Time to Maturity | ↑ | ↑ |
| ↑ Interest Rate | ↑ | ↓ |

### The Black-Scholes Assumptions
- ✓ European options (no early exercise)
- ✓ Constant volatility
- ✓ Log-normal stock prices
- ✓ No dividends
- ✓ No transaction costs
- ✓ No arbitrage opportunities